In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from cmfa.tca_openflux import (
    TCA_OpenFLux,
    objective_wssr,
    objective_clr_wssr,
    # objective_leave_one_out_wssr, # missing
    flux_vector,
    calc_ci
)
from cmfa.emu_example import closure, clr_func, clr_inv_func

import optimistix
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [2]:
def fva(parameters: dict[str, float]) -> dict:
    v_free = {"v8": 0.215, "v1": 1, "v19": 0.05, "v20": 0.03, "v16": 0.07, "v17": 0.23, "v18": 0.12, "v2": 0.201, "v9": 0.296}
    fixed_keys = ["v1", "v16", "v17", "v18", "v19", "v20"]
    fixed = {k: v_free[k] for k in fixed_keys}

    # True MIDs
    true_data = TCA_OpenFLux(parameters=parameters, fixed=fixed)

    key = jax.random.PRNGKey(1234)
    sims = 1000
    total_mol = 1e5
    total_isos    = jax.tree.map(lambda x: x * total_mol, true_data)
    norm_sigma_cv = jax.tree.map(lambda x: x * 0.1, total_isos)

    key, key_sim = jax.random.split(key)
    norm_sim = jax.tree.map(
        lambda mean, sigma: mean + sigma * jax.random.normal(key_sim, (sims, mean.shape[0])),
        total_isos, norm_sigma_cv,
    )
    iso_proportions = jax.tree.map(lambda mat: closure(mat), norm_sim)

    mean_sim = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_proportions)
    std_sim  = jax.tree.map(lambda x: jnp.std(x,  axis=0), iso_proportions)
    D_sim    = jax.tree.map(lambda x: jnp.diag(x), std_sim)

    solver = optimistix.BFGS(rtol=1e-12, atol=1e-12, verbose=True)
    init_params = {"v8": 0.2, "v2": 0.2, "v9": 0.3}

    # ── Euclidean (WSSR) ──────────────────────────────────────────────────────
    result_euc = optimistix.minimise(
        objective_wssr, y0=init_params,
        args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000,
    )

    # ── Aitchison (CLR-WSSR) ──────────────────────────────────────────────────
    iso_clr  = jax.tree.map(clr_func, iso_proportions)
    mean_clr = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_clr)
    std_clr  = jax.tree.map(lambda x: jnp.std(x,  axis=0), iso_clr)
    D_clr    = jax.tree.map(lambda x: jnp.diag(x), std_clr)

    result_ait = optimistix.minimise(
        objective_clr_wssr, y0=init_params,
        args=(mean_clr, D_clr, fixed), solver=solver, max_steps=10000,
    )

    # ── Leave-one-out WSSR ────────────────────────────────────────────────────
    init_params_loo = {
        "fluxes": {"v8": 0.2, "v2": 0.2, "v9": 0.3},
        "scales": jax.tree.map(lambda _: 1.0, mean_sim),
    }
    result_loo = optimistix.minimise(
        objective_leave_one_out_wssr, y0=init_params_loo,
        args=(mean_sim, D_sim, fixed), solver=solver, max_steps=10000,
    )

    true_mids = {k: np.array(v) for k, v in true_data.items()}

    return {
        "Euclidean":   result_euc.value,
        "Aitchison":   result_ait.value,
        "LeaveOneOut": result_loo.value["fluxes"],
        "true_mids":   true_mids,
    }

In [3]:
parameters = {"v8": [0.200, 0.210, 0.220], "v2": [0.190, 0.20, 0.210], "v9": [0.190, 0.20, 0.210]}
v_true_dict = []
v_est_dict  = []
v_clr_dict  = []
v_loo_dict  = []
for param_set in (
    dict(zip(parameters, values))
    for values in zip(*parameters.values())
):
    result = fva(param_set)
    v_true_dict.append(param_set)
    v_est_dict.append(result["Euclidean"])
    v_clr_dict.append(result["Aitchison"])
    v_loo_dict.append(result["LeaveOneOut"])

Loss on this step: 0.04332797273352411, Loss on the last accepted step: 0.0, Step size: 1.0, y: {'v2': Array(0.2, dtype=float64), 'v8': Array(0.2, dtype=float64), 'v9': Array(0.3, dtype=float64)}, y on the last accepted step: {'v2': Array(0.2, dtype=float64), 'v8': Array(0.2, dtype=float64), 'v9': Array(0.3, dtype=float64)}
Loss on this step: 7635.5005402638835, Loss on the last accepted step: 0.04332797273352411, Step size: 0.5, y: {'v2': Array(1.61134988, dtype=float64), 'v8': Array(0.14788239, dtype=float64), 'v9': Array(-0.51950069, dtype=float64)}, y on the last accepted step: {'v2': Array(0.2, dtype=float64), 'v8': Array(0.2, dtype=float64), 'v9': Array(0.3, dtype=float64)}
Loss on this step: 6117.711404794162, Loss on the last accepted step: 0.04332797273352411, Step size: 0.25, y: {'v2': Array(0.90567494, dtype=float64), 'v8': Array(0.17394119, dtype=float64), 'v9': Array(-0.10975035, dtype=float64)}, y on the last accepted step: {'v2': Array(0.2, dtype=float64), 'v8': Array(0.

NameError: name 'objective_leave_one_out_wssr' is not defined

In [ ]:
flux_vars = ["v2", "v8", "v9"]
MARKERS = {"Euclidean": "o", "Aitchison": "s", "LeaveOneOut": "^"}

true_vals = {v: np.array([d[v]        for d in v_true_dict]) for v in flux_vars}
euc_vals  = {v: np.array([float(d[v]) for d in v_est_dict])  for v in flux_vars}
ait_vals  = {v: np.array([float(d[v]) for d in v_clr_dict])  for v in flux_vars}
loo_vals  = {v: np.array([float(d[v]) for d in v_loo_dict])  for v in flux_vars}

sns.set_theme(style="whitegrid", palette="colorblind")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, v in zip(axes, flux_vars):
    t = true_vals[v]
    all_vals = np.concatenate([t, euc_vals[v], ait_vals[v], loo_vals[v]])
    lo, hi = all_vals.min(), all_vals.max()
    pad = (hi - lo) * 0.1
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, label="y = x")
    for arr, method in [(euc_vals, "Euclidean"), (ait_vals, "Aitchison"), (loo_vals, "LeaveOneOut")]:
        ax.scatter(t, arr[v], marker=MARKERS[method], s=60, label=method, zorder=3)
    ax.set_xlabel(f"True {v}")
    ax.set_ylabel(f"Estimated {v}")
    ax.set_title(f"Flux {v}")
    ax.legend(fontsize=8)

fig.suptitle("Euclidean vs Aitchison vs LeaveOneOut: estimated vs true fluxes", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Batch run: load the 30 LHS samples produced by flux_range.ipynb
# ---------------------------------------------------------------------------
flux_df = pd.read_csv("flux_sampling/flux_samples_R2_R8_R9_validated.csv", index_col="Sample")
print(f"Loaded {len(flux_df)} flux combinations:")
print(flux_df.round(4))

batch_true      = []
batch_euc       = []
batch_ait       = []
batch_loo       = []
batch_true_mids = []

for sample_id, row in flux_df.iterrows():
    params = {"v2": float(row["R2"]), "v8": float(row["R8"]), "v9": float(row["R9"])}
    result = fva(params)
    batch_true.append(params)
    batch_euc.append(result["Euclidean"])
    batch_ait.append(result["Aitchison"])
    batch_loo.append(result["LeaveOneOut"])
    batch_true_mids.append(result["true_mids"])

print(f"\nFinished {len(batch_true)} runs.")

In [ ]:
# ── Build tidy long-form DataFrame of true MIDs ─────────────────────────────
mid_records = []
mid_keys = list(batch_true_mids[0].keys())

for i, mids in enumerate(batch_true_mids):
    for key, values in mids.items():
        for iso_idx, val in enumerate(values):
            mid_records.append({
                "Sample":      i,
                "Metabolite":  key,
                "Isotopologue": f"m+{iso_idx}",
                "iso_idx":     iso_idx,
                "MID":         float(val),
            })

mid_df = pd.DataFrame(mid_records)

# ── Plot ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="colorblind")

n_keys = len(mid_keys)
fig, axes = plt.subplots(1, n_keys, figsize=(4 * n_keys, 4), sharey=True)
fig.suptitle("True MID ranges across 50 LHS samples", fontsize=12, y=1.02)

for ax, key in zip(axes, mid_keys):
    sub = mid_df[mid_df["Metabolite"] == key].sort_values("iso_idx")

    # Individual sample lines — faint spaghetti to show spread
    for sample_id, grp in sub.groupby("Sample"):
        ax.plot(
            grp["Isotopologue"], grp["MID"],
            color="steelblue", alpha=0.15, lw=1, zorder=1,
        )

    # Mean ± 1 SD band on top
    stats_df = sub.groupby(["iso_idx", "Isotopologue"])["MID"].agg(
        mean="mean", std="std"
    ).reset_index().sort_values("iso_idx")

    ax.fill_between(
        stats_df["Isotopologue"],
        (stats_df["mean"] - stats_df["std"]).clip(lower=0),
        (stats_df["mean"] + stats_df["std"]).clip(upper=1),
        alpha=0.25, color="steelblue", zorder=2,
    )
    ax.plot(
        stats_df["Isotopologue"], stats_df["mean"],
        color="steelblue", lw=2, marker="o", ms=5, zorder=3, label="Mean",
    )

    ax.set_ylim(0, 0.5)
    ax.set_title(key, fontsize=10)
    ax.set_xlabel("Isotopologue")
    ax.tick_params(axis="x", rotation=45)

axes[0].set_ylabel("MID (fraction)")

# Single legend for the whole figure
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
legend_handles = [
    Line2D([0], [0], color="steelblue", alpha=0.4, lw=1, label="Individual samples"),
    Line2D([0], [0], color="steelblue", lw=2, marker="o", ms=5, label="Mean"),
    Patch(facecolor="steelblue", alpha=0.25, label="± 1 SD"),
]
fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.0, 1.0), fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
# ── Annotate mid_df with v8 group ────────────────────────────────────────────
v8_true   = np.array([d["v8"] for d in batch_true])
v8_group  = {i: ("v8 > 0.5" if v8_true[i] > 0.5 else "v8 ≤ 0.5")
             for i in range(len(batch_true))}
mid_df["v8_group"] = mid_df["Sample"].map(v8_group)

# OAA_1234 is stored as aspx_1234 (aspx_1234 = oaa_1234 in the model)
target_mets  = {"aspx_1234": "OAA_1234", "suc_1234": "SUC_1234"}
GROUP_COLORS = {"v8 > 0.5": sns.color_palette("colorblind")[1],   # orange
                "v8 ≤ 0.5": sns.color_palette("colorblind")[0]}   # blue

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
fig.suptitle("True MID profiles split by v8 flux (> 0.5 vs ≤ 0.5)", fontsize=12, y=1.02)

for ax, (met_key, met_label) in zip(axes, target_mets.items()):
    sub = mid_df[mid_df["Metabolite"] == met_key].sort_values("iso_idx")

    # ── Spaghetti lines per sample ───────────────────────────────────────────
    for sample_id, grp in sub.groupby("Sample"):
        colour = GROUP_COLORS[grp["v8_group"].iloc[0]]
        ax.plot(grp["Isotopologue"], grp["MID"],
                color=colour, alpha=0.2, lw=1, zorder=1)

    # ── Mean line per group ───────────────────────────────────────────────────
    for group, colour in GROUP_COLORS.items():
        grp_mean = (sub[sub["v8_group"] == group]
                    .groupby(["iso_idx", "Isotopologue"])["MID"]
                    .mean()
                    .reset_index()
                    .sort_values("iso_idx"))
        ax.plot(grp_mean["Isotopologue"], grp_mean["MID"],
                color=colour, lw=2.5, marker="o", ms=6,
                zorder=3, label=f"{group}  (mean)")

    ax.set_ylim(0, 1)
    ax.set_title(met_label, fontsize=11)
    ax.set_xlabel("Isotopologue")
    ax.tick_params(axis="x", rotation=45)

axes[0].set_ylabel("MID (fraction)")

# ── Shared legend ─────────────────────────────────────────────────────────────
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
legend_handles = [
    Line2D([0], [0], color=GROUP_COLORS["v8 > 0.5"], lw=2.5, marker="o", ms=6, label="v8 > 0.5  (mean)"),
    Patch(facecolor=GROUP_COLORS["v8 > 0.5"], alpha=0.2, label="v8 > 0.5  (samples)"),
    Line2D([0], [0], color=GROUP_COLORS["v8 ≤ 0.5"], lw=2.5, marker="o", ms=6, label="v8 ≤ 0.5  (mean)"),
    Patch(facecolor=GROUP_COLORS["v8 ≤ 0.5"], alpha=0.2, label="v8 ≤ 0.5  (samples)"),
]
fig.legend(handles=legend_handles, loc="upper right",
           bbox_to_anchor=(1.15, 1.0), fontsize=9)

fig.tight_layout()
plt.show()

# ── Count per group ───────────────────────────────────────────────────────────
counts = mid_df.drop_duplicates("Sample")["v8_group"].value_counts()
print("Samples per group:", dict(counts))

## Why do the three methods fail differently? A mechanistic explanation

### The shared root problem: a saturating forward model

Every method estimates v8 by inverting the forward map **flux → MID**. That map enters through the EMU steady-state balance, where v8 appears only as the ratio **v8 / (v6 + v8)** (e.g. `A4` row 1: `[v8, -v6-v8, v6, 0]`). As v8 grows, that ratio approaches 1 and stops changing — the forward map saturates. The gradient of any objective with respect to v8 therefore *shrinks as v8 increases*, making v8 harder to identify for all three methods at high v8. The sensitivity plot below (`∂MID/∂v8`) quantifies this directly.

---

### Why Euclidean is noisy and biased *in general* (not just at high v8)

MIDs are **compositions**: they live on the simplex and sum to 1 by definition. This imposes an exact linear constraint, which means the true variance–covariance matrix of a composition is **rank-deficient** (only n−1 components are free). The Euclidean WSSR in the code uses a **diagonal** D matrix — it treats all n isotopologues as independent, which is structurally wrong in two ways:

1. **Double-counting**: Using all n components over-counts the available information (effectively treating n independent measurements instead of n−1). This inflates apparent certainty, pushing the optimizer into false confidence about v8.
2. **Asymmetric weighting at the boundary**: When a component is near zero its variance σ² → 0, so the weight 1/σ² → ∞. A small absolute deviation in that near-zero component dominates the objective even though it carries no more biological information. This distorts the optimization landscape asymmetrically depending on which isotopologue happens to be near zero.

The Leave-One-Out method directly addresses point 1 by **dropping the last isotopologue** (resolving the redundancy) and by jointly estimating a per-metabolite scale factor. It works in a geometrically correct (n−1)-dimensional subspace, which is why its residuals are typically smaller and less biased than pure Euclidean.

---

### Why Aitchison also degrades at high v8

The CLR transform `clr(x)_i = log(x_i) − mean(log(x))` rescues the near-zero problem: a fraction of 0.01 maps to a large *negative* CLR value, maintaining gradient signal where Euclidean goes blind. This is why Aitchison outperforms Euclidean at intermediate v8 where some isotopologues are near zero.

But CLR cannot fix the saturating Jacobian. At high v8, **all** suc_1234 isotopologues become moderate (no near-zeros), so CLR gains nothing over Euclidean in terms of amplifying signal. Meanwhile the true gradient `∂MID/∂v8` is tiny because the ratio v8/(v6+v8) has saturated. The CLR Jacobian is `J_CLR(MID) · ∂MID/∂v8`, and when `∂MID/∂v8 ≈ 0`, even CLR cannot recover the signal. This is confirmed by the sensitivity plot: both `‖∂MID/∂v8‖` and `‖∂CLR(MID)/∂v8‖` converge toward zero at large v8.

---

### Summary table

| Method | Fixes simplex geometry | Fixes boundary (near-zero) amplification | Fixes v8 saturation at high flux |
|---|---|---|---|
| Euclidean | ✗ | ✗ | ✗ |
| Aitchison (CLR) | ✓ | ✓ | ✗ |
| Leave-One-Out | Partially (drops last) | ✗ | ✗ |

None of the three methods can recover v8 when the forward model saturates — that is a fundamental **identifiability limit** of the experiment, not a property of the distance metric.

In [ ]:
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

fixed_sens = {"v1": 1.0, "v16": 0.07, "v17": 0.23, "v18": 0.12, "v19": 0.05, "v20": 0.03}
BASE_V2, BASE_V9 = 0.2, 0.3

# ── JAX Jacobian: ∂MID / ∂v8 for each metabolite ────────────────────────────
def mid_of_v8(v8_val):
    params = {"v2": BASE_V2, "v8": v8_val, "v9": BASE_V9}
    return TCA_OpenFLux(params, fixed_sens)

jac_fn = jax.jacobian(mid_of_v8)   # returns dict of arrays shape (n_iso,)

# CLR Jacobian of the CLR transform: J_clr(x)_ij = (δ_ij - 1/n) / x_i
def clr_jacobian(x):
    n = x.shape[0]
    return jnp.diag(1.0 / x) @ (jnp.eye(n) - jnp.ones((n, n)) / n)

v8_range = jnp.linspace(0.01, 0.99, 150)

sens_records = []
for v8_val in v8_range:
    mids = mid_of_v8(v8_val)
    jac  = jac_fn(v8_val)           # dict: key → array of shape (n_iso,)

    for met_key in mids.keys():
        mid_vec  = mids[met_key]     # shape (n,)
        dmid_dv8 = jac[met_key]      # shape (n,)

        # Euclidean sensitivity: ‖∂MID/∂v8‖₂
        sens_euc = float(jnp.linalg.norm(dmid_dv8))

        # Aitchison sensitivity: ‖J_CLR(MID) · ∂MID/∂v8‖₂
        J_clr     = clr_jacobian(mid_vec)
        dclr_dv8  = J_clr @ dmid_dv8
        sens_ait  = float(jnp.linalg.norm(dclr_dv8))

        # Leave-One-Out sensitivity: ‖∂MID[:-1]/∂v8‖₂  (drop last isotopologue)
        sens_loo  = float(jnp.linalg.norm(dmid_dv8[:-1]))

        sens_records.append({
            "v8":        float(v8_val),
            "Metabolite": met_key,
            "Euclidean":  sens_euc,
            "Aitchison":  sens_ait,
            "LeaveOneOut": sens_loo,
        })

sens_df = pd.DataFrame(sens_records)
sens_long = sens_df.melt(
    id_vars=["v8", "Metabolite"],
    value_vars=METHODS,
    var_name="Method", value_name="Sensitivity",
)

# ── Plot ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="colorblind")
met_labels = {"lysx_123456": "LysX (123456)", "valx_12345": "ValX (12345)",
              "aspx_1234": "OAA / AspX (1234)", "suc_1234": "SUC (1234)"}

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
fig.suptitle(
    "Sensitivity ‖∂MID/∂v8‖ vs true v8  —  how much each method can 'see' v8 changes",
    fontsize=11, y=1.02,
)

for ax, met_key in zip(axes, list(met_labels.keys())):
    sub = sens_long[sens_long["Metabolite"] == met_key]
    sns.lineplot(
        data=sub, x="v8", y="Sensitivity",
        hue="Method", hue_order=METHODS,
        palette="colorblind", lw=2, ax=ax,
    )
    ax.axvline(0.5, color="grey", lw=1, ls=":", alpha=0.7, label="v8 = 0.5")
    ax.set_title(met_labels[met_key], fontsize=9)
    ax.set_xlabel("True v8")
    ax.set_ylabel("‖∂MID/∂v8‖₂")
    if ax != axes[0]:
        ax.get_legend().remove()

axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

# ── Ratio: Aitchison / Euclidean sensitivity  ────────────────────────────────
# Values > 1 mean Aitchison "sees" v8 more sharply than Euclidean at that v8
ratio_df = sens_df.copy()
ratio_df["Ait/Euc ratio"] = ratio_df["Aitchison"] / ratio_df["Euclidean"].replace(0, np.nan)
ratio_df["LOO/Euc ratio"] = ratio_df["LeaveOneOut"] / ratio_df["Euclidean"].replace(0, np.nan)
ratio_long = ratio_df.melt(
    id_vars=["v8", "Metabolite"],
    value_vars=["Ait/Euc ratio", "LOO/Euc ratio"],
    var_name="Ratio", value_name="Value",
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
fig.suptitle(
    "Sensitivity gain vs Euclidean  (>1 = method sees more signal than Euclidean)",
    fontsize=11, y=1.02,
)
for ax, met_key in zip(axes, list(met_labels.keys())):
    sub = ratio_long[ratio_long["Metabolite"] == met_key]
    sns.lineplot(data=sub, x="v8", y="Value", hue="Ratio",
                 palette=["#d62728", "#ff7f0e"], lw=2, ax=ax)
    ax.axhline(1, color="black", lw=1, ls="--")
    ax.axvline(0.5, color="grey", lw=1, ls=":", alpha=0.7)
    ax.set_title(met_labels[met_key], fontsize=9)
    ax.set_xlabel("True v8")
    ax.set_ylabel("Sensitivity ratio")
    if ax != axes[0]:
        ax.get_legend().remove()

axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
flux_vars = ["v2", "v8", "v9"]
MARKERS   = {"Euclidean": "o", "Aitchison": "s", "LeaveOneOut": "^"}
METHODS   = ["Euclidean", "Aitchison", "LeaveOneOut"]

true_arr = {v: np.array([d[v]        for d in batch_true]) for v in flux_vars}
euc_arr  = {v: np.array([float(d[v]) for d in batch_euc])  for v in flux_vars}
ait_arr  = {v: np.array([float(d[v]) for d in batch_ait])  for v in flux_vars}
loo_arr  = {v: np.array([float(d[v]) for d in batch_loo])  for v in flux_vars}

# ── Build tidy long-form DataFrame ──────────────────────────────────────────
records = []
for i in range(len(batch_true)):
    for v in flux_vars:
        for method, arr in [("Euclidean", euc_arr), ("Aitchison", ait_arr), ("LeaveOneOut", loo_arr)]:
            records.append({
                "Sample":    i,
                "Flux":      v,
                "Method":    method,
                "True":      true_arr[v][i],
                "Estimated": float(arr[v][i]),
                "Residual":  float(arr[v][i]) - true_arr[v][i],
            })

df = pd.DataFrame(records)

# ── IQR outlier detection (worst |residual| across all methods and fluxes) ──
abs_res = df.groupby("Sample")["Residual"].apply(lambda x: np.abs(x).max())
q1, q3  = abs_res.quantile(0.25), abs_res.quantile(0.75)
iqr     = q3 - q1
outlier_samples = set(abs_res[abs_res > q3 + 1.5 * iqr].index.tolist())

df["Outlier"] = df["Sample"].isin(outlier_samples)
df_clean   = df[~df["Outlier"]]
df_outlier = df[df["Outlier"]]
n_out = len(outlier_samples)
print(f"Outlier samples (IQR): {sorted(outlier_samples)}  ({n_out}/{len(batch_true)} removed)")

# ── Plot ─────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="colorblind")
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(
    f"Euclidean vs Aitchison vs LeaveOneOut — {len(batch_true) - n_out} clean samples "
    f"({n_out} outlier{'s' if n_out != 1 else ''} removed by IQR)",
    fontsize=12, y=1.01,
)
palette = sns.color_palette("colorblind", n_colors=3)

for col, v in enumerate(flux_vars):
    sub = df_clean[df_clean["Flux"] == v]

    # ── Row 0: parity ─────────────────────────────────────────────────────────
    ax = axes[0, col]
    lo = min(sub["True"].min(), sub["Estimated"].min())
    hi = max(sub["True"].max(), sub["Estimated"].max())
    pad = (hi - lo) * 0.05
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
            color="black", lw=1, ls="--", zorder=0)
    sns.scatterplot(
        data=sub, x="True", y="Estimated",
        hue="Method", style="Method", markers=MARKERS,
        hue_order=METHODS, style_order=METHODS,
        s=50, alpha=0.85, ax=ax, legend=(col == 0),
    )
    ax.set_title(f"{v}  —  parity")
    ax.set_xlabel(f"True {v}")
    ax.set_ylabel(f"Estimated {v}")
    if col == 0:
        ax.legend(fontsize=8)

    # ── Row 1: residuals ──────────────────────────────────────────────────────
    ax = axes[1, col]
    ax.axhline(0, color="black", lw=1, ls="--", zorder=0)
    sns.scatterplot(
        data=sub, x="True", y="Residual",
        hue="Method", style="Method", markers=MARKERS,
        hue_order=METHODS, style_order=METHODS,
        s=50, alpha=0.85, ax=ax, legend=False,
    )
    for method, colour in zip(METHODS, palette):
        bias = sub.loc[sub["Method"] == method, "Residual"].mean()
        ax.axhline(bias, color=colour, lw=1.2, ls=":", alpha=0.8, label=f"{method} bias")
    ax.set_title(f"{v}  —  residuals")
    ax.set_xlabel(f"True {v}")
    ax.set_ylabel("Residual (estimated − true)")
    if col == 0:
        ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

# ── Summary statistics ────────────────────────────────────────────────────────
rows = []
for v in flux_vars:
    for method in METHODS:
        err = df_clean.loc[(df_clean["Flux"] == v) & (df_clean["Method"] == method), "Residual"].values
        rows.append({
            "Flux": v, "Method": method,
            "Bias": float(err.mean()),
            "MAE":  float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        })

summary = pd.DataFrame(rows).set_index(["Flux", "Method"])
pd.set_option("display.float_format", "{:.5f}".format)
print(f"\nSummary statistics ({len(batch_true) - n_out} clean samples):")
print(summary)

if n_out:
    print(f"\nOutlier sample details (excluded):")
    print(df_outlier.pivot_table(
        index="Sample", columns=["Flux", "Method"], values="Residual"
    ).round(4))

## Why do outlier samples produce extreme v8 errors under Euclidean distance?

### Root cause 1 — Succinate symmetry creates a flat Euclidean objective for v8

`v8` enters every EMU transition matrix with a **0.5 multiplier** (e.g. `A1` rows 6, 8, 11, 16; `A3` rows 3–4; `A4` row 1 in `tca_openflux.py`).  
These 0.5 terms encode the **symmetric scrambling of succinate**: because succinate has C₂ symmetry, carbons 1–2 and 3–4 are interchangeable, and the label information in the measured MIDs is halved.  
More critically, `v8` only appears in the steady-state EMU balance as the **ratio `v8 / (v6 + v8)`** (diagonal term `−v6−v8`, off-diagonal `v8`). At large v8 values (as sampled by LHS: v8 up to 1.0), this ratio saturates toward 1 and becomes insensitive to further increases in v8. The Euclidean WSSR objective therefore develops a **flat plateau** along the v8 axis, and BFGS drifts to an arbitrary value.

### Root cause 2 — Euclidean distance ignores the simplex geometry

MIDs are **compositions** (they sum to 1 and live on the simplex). The Euclidean WSSR treats them as unconstrained points in ℝⁿ.  
When some isotopologue fractions are near zero (which happens for extreme flux values), the variance-scaling in WSSR collapses those terms (σ ≈ 0 → down-weight → near-zero gradient). The optimizer loses signal precisely where the data would be most informative.

### Why Aitchison distance is more robust

The CLR transform maps the simplex to ℝⁿ via **log-ratios**, so a near-zero fraction (e.g. M+0 ≈ 0.01) maps to a large *negative* CLR value — not a small one. The Aitchison objective therefore maintains a strong gradient everywhere on the simplex, including near the boundary, which keeps BFGS anchored near the true v8.

### Additional analyses below

1. **Objective landscape** — 1-D slice of WSSR vs CLR-WSSR along the v8 axis for an outlier sample, directly showing the flat Euclidean plateau.  
2. **Recovery rate** — fraction of samples that recover the true flux within ±5 %, ±10 %, ±20 %.  
3. **Error correlation heatmap** — whether errors in v2/v8/v9 are correlated, exposing the joint identifiability structure.

In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")
palette = sns.color_palette("colorblind", n_colors=3)

fixed_keys = ["v1", "v16", "v17", "v18", "v19", "v20"]
v_free_ref  = {"v8": 0.215, "v1": 1, "v19": 0.05, "v20": 0.03,
               "v16": 0.07, "v17": 0.23, "v18": 0.12, "v2": 0.201, "v9": 0.296}
fixed = {k: v_free_ref[k] for k in fixed_keys}

def simulate(params):
    true_data   = TCA_OpenFLux(parameters=params, fixed=fixed)
    total_isos  = jax.tree.map(lambda x: x * 1e5, true_data)
    cv_sigma    = jax.tree.map(lambda x: x * 0.1,  total_isos)
    key         = jax.random.PRNGKey(1234)
    _, key_sim  = jax.random.split(key)
    norm_sim    = jax.tree.map(
        lambda m, s: m + s * jax.random.normal(key_sim, (1000, m.shape[0])),
        total_isos, cv_sigma)
    iso_prop    = jax.tree.map(lambda x: closure(x), norm_sim)
    mean_s      = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_prop)
    std_s       = jax.tree.map(lambda x: jnp.std(x,  axis=0), iso_prop)
    D_s         = jax.tree.map(lambda x: jnp.diag(x), std_s)
    iso_clr     = jax.tree.map(clr_func, iso_prop)
    mean_clr    = jax.tree.map(lambda x: jnp.mean(x, axis=0), iso_clr)
    std_clr     = jax.tree.map(lambda x: jnp.std(x,  axis=0), iso_clr)
    D_clr       = jax.tree.map(lambda x: jnp.diag(x), std_clr)
    return mean_s, D_s, mean_clr, D_clr

# ═══════════════════════════════════════════════════════════════════════════════
# 1. OBJECTIVE LANDSCAPE — 1-D v8 slice for one outlier sample
# ═══════════════════════════════════════════════════════════════════════════════
outlier_idx = max(outlier_samples, key=lambda i: batch_true[i]["v8"])
true_params = batch_true[outlier_idx]
print(f"Outlier sample {outlier_idx}: true params = {true_params}")

mean_s, D_s, mean_clr, D_clr = simulate(true_params)

v8_sweep = np.linspace(0.01, 1.0, 200)
wssr_vals, clr_vals, loo_vals_obj = [], [], []

for v8_val in v8_sweep:
    p = {**true_params, "v8": float(v8_val)}
    wssr_vals.append(float(objective_wssr(p, (mean_s, D_s, fixed))))
    clr_vals.append( float(objective_clr_wssr(p, (mean_clr, D_clr, fixed))))
    p_loo = {"fluxes": p, "scales": jax.tree.map(lambda _: 1.0, mean_s)}
    loo_vals_obj.append(float(objective_leave_one_out_wssr(p_loo, (mean_s, D_s, fixed))))

landscape_df = pd.DataFrame({
    "v8":        np.tile(v8_sweep, 3),
    "Objective": np.concatenate([wssr_vals, clr_vals, loo_vals_obj]),
    "Method":    ["Euclidean"] * 200 + ["Aitchison"] * 200 + ["LeaveOneOut"] * 200,
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f"Objective landscape along v8 axis  |  outlier sample {outlier_idx} "
    f"(true v8 = {true_params['v8']:.3f})",
    fontsize=11,
)
for ax, (method, colour) in zip(axes, zip(METHODS, palette)):
    sub = landscape_df[landscape_df["Method"] == method]
    ax.plot(sub["v8"], sub["Objective"], color=colour, lw=1.8)
    ax.axvline(true_params["v8"], color="black", lw=1.2, ls="--", label="True v8")
    est_v8 = (float(batch_euc[outlier_idx]["v8"])  if method == "Euclidean" else
              float(batch_ait[outlier_idx]["v8"])  if method == "Aitchison" else
              float(batch_loo[outlier_idx]["v8"]))
    ax.axvline(est_v8, color=colour, lw=1.2, ls=":", label="Estimated v8")
    ax.set_xlabel("v8")
    ax.set_ylabel("Objective value")
    ax.set_title(method)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 2. RECOVERY RATE
# ═══════════════════════════════════════════════════════════════════════════════
tolerances = [0.05, 0.10, 0.20]
recovery_rows = []
for v in flux_vars:
    for method in METHODS:
        sub = df_clean[(df_clean["Flux"] == v) & (df_clean["Method"] == method)]
        rel_err = (sub["Residual"] / sub["True"]).abs()
        for tol in tolerances:
            recovery_rows.append({
                "Flux": v, "Method": method,
                "Tolerance": f"±{int(tol*100)}%",
                "Recovery (%)": round(100 * (rel_err <= tol).mean(), 1),
            })

recovery_df = pd.DataFrame(recovery_rows)
print("Recovery rate (% of clean samples within relative tolerance):")
print(recovery_df.pivot_table(
    index=["Flux", "Method"], columns="Tolerance", values="Recovery (%)"
).to_string())

g = sns.FacetGrid(recovery_df, col="Tolerance", height=4, aspect=0.85, sharey=True)
g.map_dataframe(sns.barplot, x="Flux", y="Recovery (%)", hue="Method",
                hue_order=METHODS, order=flux_vars,
                palette="colorblind", errorbar=None)
g.add_legend()
g.set_axis_labels("Flux", "Recovery (%)")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("Recovery rate by tolerance threshold (clean samples)", y=1.03)
for ax in g.axes.flat:
    ax.set_ylim(0, 105)
    ax.axhline(100, color="grey", lw=0.8, ls="--")
g.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3. ERROR CORRELATION HEATMAP
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Residual correlation across fluxes (clean samples)", fontsize=11)

for ax, (method, colour) in zip(axes, zip(METHODS, palette)):
    sub = df_clean[df_clean["Method"] == method].pivot(
        index="Sample", columns="Flux", values="Residual"
    )[flux_vars]
    corr = sub.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, ax=ax, annot=True, fmt=".2f", vmin=-1, vmax=1,
        cmap="coolwarm", square=True, linewidths=0.5,
        mask=mask, cbar=(ax == axes[-1]),
    )
    ax.set_title(method)
    ax.set_xlabel("")
    ax.set_ylabel("")

fig.tight_layout()
plt.show()